# Minimal cross-store query — Netherlands 2024

Pull **CO2** from the ICOS Obspack zarr, **monthly NEE** from the
FLUXNET zarr, and **fCO₂** from the ICOS Ocean (SOCAT) zarr for
stations / cruises inside a Netherlands bounding box, restricted to
2024.

- **Region**: lat 50.7–53.6, lon 3.3–7.3
- **Time**: 2024-01-01 → 2024-12-31

Each store has a *combined-view group* (built by `combine_to_dim.py`)
where the per-station / per-cruise data shares one indexed dimension,
so spatial and temporal selection becomes a single xarray expression.

| Store | Combined group | Index dim | Spatial coords |
|---|---|---|---|
| `icos-obspack.zarr` | `co2`, `ch4`, `n2o`, `co` | `station` | `lat(station)`, `lon(station)`, `intake_height(station)` |
| `icos-fluxnet.zarr` | `_combined/fluxnet_dd` … `_yy` | `station` | `lat(station)`, `lon(station)`, `station_elevation(station)` |
| `icos-socat.zarr`   | `_combined`               | `deployment` | `lat(deployment, time)`, `lon(deployment, time)` (SOCAT lat/lon vary along the cruise) |

The SOCAT combined view is built by `combine_to_dim.py socat`; QC-failed
rows (WOCE flag > 2 on the variable's QC or on the position QC) are
NaN-padded at build time, so consumers don't need to re-mask.

In [ ]:
import xarray as xr

LAT_MIN, LAT_MAX = 50.7, 53.6
LON_MIN, LON_MAX = 3.3, 7.3
T0, T1 = "2024-01-01", "2024-12-31"

## Obspack — CO2

In [ ]:
ds = xr.open_zarr("zarr//icos-obspack.zarr", group="co2", consolidated=False)

co2_nl = (
    ds["co2"]
      .where(
          (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
          (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
          drop=True,
      )
      .sel(time_co2=slice(T0, T1))
)

# Per-station summary — coords travel with the DataArray
for sid in co2_nl.station.values:
    da = co2_nl.sel(station=sid)
    print(f"{sid:8s} lat={float(co2_nl.lat.sel(station=sid)):.3f} "
          f"lon={float(co2_nl.lon.sel(station=sid)):.3f}  "
          f"intake_height={float(co2_nl.intake_height.sel(station=sid)):>4.0f} m  "
          f"n={int(da.count())}  "
          f"mean={float(da.mean()):.2f} ppm")

## FLUXNET — monthly NEE

Selects the VUT/REF NEE variant from each site's `fluxnet_mm` sub-group.

In [ ]:
ds = xr.open_zarr("zarr//icos-fluxnet.zarr", group="_combined/fluxnet_mm", consolidated=False)

nee_nl = (
    ds["NEE"]
      .sel(ustar_threshold="VUT", nee_variant="REF")
      .where(
          (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
          (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
          drop=True,
      )
      .sel(time=slice(T0, T1))
)

units = ds["NEE"].attrs.get("units", "")
for sid in nee_nl.station.values:
    da = nee_nl.sel(station=sid)
    print(f"{sid:8s} lat={float(nee_nl.lat.sel(station=sid)):.3f} "
          f"lon={float(nee_nl.lon.sel(station=sid)):.3f}  "
          f"n={int(da.count())}  "
          f"mean={float(da.mean()):.3f} {units}")

## SOCAT — fCO₂

Same pattern as the other two stores: open the `_combined` group, mask
on `lat`/`lon` (which are 2-D `(deployment, time)` here because SOCAT
lat/lon vary along each cruise — xarray broadcasts the mask correctly),
slice on `time`.

In [ ]:
import numpy as np

ds = xr.open_zarr("zarr//icos-socat.zarr", group="_combined", consolidated=False)

fco2_nl = (
    ds["fCO2"]
      .where(
          (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
          (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
          drop=True,
      )
      .sel(time=slice(T0, T1))
)

units = ds["fCO2"].attrs.get("units", "µatm")
valid = fco2_nl.values[np.isfinite(fco2_nl.values)]
print(f"SOCAT fCO2 — {fco2_nl.sizes.get('deployment', 0)} contributing deployments × "
      f"{fco2_nl.sizes.get('time', 0)} timestamps  →  "
      f"{valid.size} QC-passing samples")

if valid.size:
    print(f"\npooled mean fCO2 = {valid.mean():.1f} {units}  "
          f"(range {valid.min():.1f}–{valid.max():.1f})")
    # Per-deployment summary
    for dep in fco2_nl.deployment.values:
        da = fco2_nl.sel(deployment=dep)
        v = da.values[np.isfinite(da.values)]
        if not v.size:
            continue
        station = str(ds.station_id.sel(deployment=dep).values)
        print(f"  {str(dep):14s}  {station:30s}  n={v.size:>5d}  mean={v.mean():.1f} {units}")

## Same query via the data-passport proxy (port 8080)

Reads the stores over HTTP through `run_proxy.py`. Each `.values` access
records chunk fetches; on session close a passport is minted.

The proxy is assumed to run at https://zarr.icos-cp.eu

In [ ]:
import json
from datapassport_zarr import open_zarr

OBSPACK_URL = "https://zarr.icos-cp.eu/icos-obspack.zarr"
FLUXNET_URL = "https://zarr.icos-cp.eu/icos-fluxnet.zarr"

# Same xarray expressions as above — over HTTP, with passports.
# .load() forces the lat/lon coord chunks to fetch eagerly while keeping
# them as xarray DataArrays (with dims), so .where(..., drop=True) accepts
# the boolean mask. Without the eager fetch, lazy reads of lat/lon
# mid-mask can flake on slow / SSH-tunneled connections.
# record_query() captures bbox + surviving stations explicitly so the
# passport JSON shows them.

bbox = {"lat": [LAT_MIN, LAT_MAX], "lon": [LON_MIN, LON_MAX]}

print("=== Obspack — CO2 ===")
with open_zarr(OBSPACK_URL, group="co2", verbose=True) as ds_co2:
    ds_co2.record_query({"bbox": bbox, "time": {"time_co2": [T0, T1]}})
    ds_co2._ds["lat"].load()
    ds_co2._ds["lon"].load()
    co2_nl = (
        ds_co2["co2"]
              .where(
                  (ds_co2.lat >= LAT_MIN) & (ds_co2.lat <= LAT_MAX) &
                  (ds_co2.lon >= LON_MIN) & (ds_co2.lon <= LON_MAX),
                  drop=True,
              )
              .sel(time_co2=slice(T0, T1))
              .compute()
    )
    ds_co2.record_query({"surviving_stations": [
        s.decode() if isinstance(s, (bytes, bytearray)) else str(s)
        for s in co2_nl.station.values
    ]})
    print(f"Obspack CO2 — {len(co2_nl.station)} stations × {len(co2_nl.time_co2)} timestamps")
    print(f"  mean across NL+2024 = {float(co2_nl.mean()):.2f} ppm")

obspack_passport = ds_co2._passport

print("\n=== Fluxnet — monthly NEE ===")
with open_zarr(FLUXNET_URL, group="_combined/fluxnet_mm", verbose=True) as ds_nee:
    ds_nee.record_query({"bbox": bbox, "time": {"time": [T0, T1]},
                         "select": {"ustar_threshold": "VUT", "nee_variant": "REF"}})
    ds_nee._ds["lat"].load()
    ds_nee._ds["lon"].load()
    nee_nl = (
        ds_nee["NEE"]
              .sel(ustar_threshold="VUT", nee_variant="REF")
              .where(
                  (ds_nee.lat >= LAT_MIN) & (ds_nee.lat <= LAT_MAX) &
                  (ds_nee.lon >= LON_MIN) & (ds_nee.lon <= LON_MAX),
                  drop=True,
              )
              .sel(time=slice(T0, T1))
              .compute()
    )
    ds_nee.record_query({"surviving_stations": [
        s.decode() if isinstance(s, (bytes, bytearray)) else str(s)
        for s in nee_nl.station.values
    ]})
    print(f"Fluxnet NEE — {len(nee_nl.station)} stations × {len(nee_nl.time)} months")
    print(f"  mean across NL+2024 = {float(nee_nl.mean()):.3f} {ds_nee['NEE'].attrs.get('units','')}")

fluxnet_passport = ds_nee._passport

print("\n──── Obspack passport ─────────────────────────────────────────────────")
print(json.dumps(obspack_passport, indent=2, default=str))

print("\n──── Fluxnet passport ─────────────────────────────────────────────────")
print(json.dumps(fluxnet_passport, indent=2, default=str))

import pathlib
saved = sorted(pathlib.Path(".passport").glob("*.json"))[-2:] if pathlib.Path(".passport").exists() else []
if saved:
    print(f"\nSaved passport files: {[str(p) for p in saved]}")

### SOCAT via the proxy

With the `_combined` group in place, SOCAT becomes a single xarray
expression — same pattern as Obspack and Fluxnet above.

In [ ]:
SOCAT_URL = "https://zarr.icos-cp.eu/icos-socat.zarr"

print("=== SOCAT — fCO2 ===")
with open_zarr(SOCAT_URL, group="_combined", verbose=True) as ds_fco2:
    ds_fco2.record_query({"bbox": bbox, "time": {"time": [T0, T1]}})
    ds_fco2._ds["lat"].load()
    ds_fco2._ds["lon"].load()
    fco2_nl = (
        ds_fco2["fCO2"]
              .where(
                  (ds_fco2.lat >= LAT_MIN) & (ds_fco2.lat <= LAT_MAX) &
                  (ds_fco2.lon >= LON_MIN) & (ds_fco2.lon <= LON_MAX),
                  drop=True,
              )
              .sel(time=slice(T0, T1))
              .compute()
    )
    valid = fco2_nl.values[np.isfinite(fco2_nl.values)]
    surviving = [str(d) for d in fco2_nl.deployment.values]
    ds_fco2.record_query({"surviving_deployments": surviving,
                          "qc_passing_samples":   int(valid.size)})
    print(f"SOCAT fCO2 — {len(surviving)} deployments × "
          f"{fco2_nl.sizes.get('time', 0)} timestamps  →  "
          f"{valid.size} QC-passing samples")
    if valid.size:
        print(f"  pooled mean fCO2 = {valid.mean():.1f} µatm")

socat_passport = ds_fco2._passport

print("\n──── SOCAT passport ──────────────────────────────────────────────────")
print(json.dumps(socat_passport, indent=2, default=str))